Imports

In [3]:
import pandas as pd

Read Raw Data

In [4]:
mbs_raw_data_df = pd.read_csv('../data/raw/mbs_dataset_raw_sample.csv', 
                              dtype={"tax_ref_no": str,"natlidno": str,"mobilephone": str, "pr_mbr_no": str}
                              )

Validation

In [5]:
# Ndabezinhle fix

from validation import validate_all

# validate_all(mbs_raw_data_df)

Read Validated Data

In [7]:
# mbs_data_df = pd.read_csv('../data/primary/mbs_dataset_validated.csv', 
#                             dtype={"tax_ref_no": str, "natlidno": str, "mobilephone": str, "pr_mbr_no": str}
#                             )

mbs_data_df = mbs_raw_data_df # Remove once validation code is complete

Generate Statement for each Member

In [8]:
from pathlib import Path

from PIL import Image
from statement_templates.saccawu import generate

old_mutual_logo = Image.open("../assets/old_mutual_header.png")
saccawu_logo = Image.open("../assets/saccawu_logo.png")

reporting_dt = mbs_data_df[mbs_data_df["acc_credit"].notna()]["pyrl_dt"].max()
reporting_dt = pd.to_datetime(reporting_dt)

start_dt = reporting_dt.replace(day=1) - pd.DateOffset(months=11)

# Loop through members to process each members statement
for case_mbr_key in mbs_data_df["case_mbr_key"].dropna().unique():
    
    # Filter mbs data between a period and current member
    mbs_data_df["pyrl_dt"] = pd.to_datetime(mbs_data_df["pyrl_dt"])
    mbs_data = mbs_data_df[mbs_data_df["pyrl_dt"].between(start_dt, reporting_dt) & (mbs_data_df["case_mbr_key"] == case_mbr_key)]
        
    output_path = Path(f"../statements/{case_mbr_key}_member_statement.pdf")

    generate.generate_statement(mbs_data, output_path, old_mutual_logo, saccawu_logo, reporting_dt, start_dt)

In [42]:
import importlib
import post_validation

importlib.reload(post_validation)

<module 'post_validation' from 'c:\\Users\\X525637\\OneDrive - Old Mutual\\Documents\\MBS 2026\\MBS-Automation\\src\\post_validation.py'>

In [ ]:
from builders import transactions_builder, investment_builder, benefits_builder, salary_builder
from builders.statement_builder import build_statement_context
from post_validation import (extract_pdf_metadata, validate_member_fields, validate_page_count, validate_pdf_word_counts, validate_salary, validate_contribution_history, validate_investment, validate_benefits) 

#)

expected_counts = {
    1: 297,
    2: 190,
    3: 260, 
    4: 415,
    5: 554,
    6: 527,
    7: 217
}

page_words_results = []
pdf_page_count_results = []
member_validation_results = []
salary_validation_results = []
transaction_validation_results = []
investment_validation_results = []
benefits_validation_results = []

for case_mbr_key in mbs_data_df["case_mbr_key"].dropna().unique():

    member_data = mbs_data_df[(mbs_data_df["pyrl_dt"].between(start_dt, reporting_dt)) & (mbs_data_df["case_mbr_key"] == case_mbr_key)]
    member_row = member_data.iloc[-1]
    pdf_path = f"../statements/{case_mbr_key}_member_statement.pdf"
    pdf_data = extract_pdf_metadata(pdf_path)
    
    context = build_statement_context(member_data)

    page_results = validate_page_count(
        case_mbr_key=case_mbr_key,
        expected_pages=7,
        actual_pages=pdf_data["page_count"])

    page_count = validate_pdf_word_counts(case_mbr_key=case_mbr_key, pdf_path=pdf_path, expected_counts=expected_counts)

    member_results = validate_member_fields(
        case_mbr_key=case_mbr_key,
        member_row=member_row,
        pdf_text=pdf_data["full_text"])

    salary_results = validate_salary(
        case_mbr_key=case_mbr_key,
        salary=context.salary,
        pdf_text=pdf_data["full_text"])

    acc_transaction_results = validate_contribution_history(
        case_mbr_key=case_mbr_key,
        contribution_history=context.contributions,
        transaction_summary=context.transactions,
        pdf_text=pdf_data["full_text"])

    investment_results = validate_investment(
        case_mbr_key=case_mbr_key,
        investment_portfolio=context.investments,
        investment_pot=context.investments,
        pdf_text=pdf_data["full_text"])

    benefits_results = validate_benefits(
        case_mbr_key=case_mbr_key,
        benefits=context.benefits,
        pdf_text=pdf_data["full_text"])


    page_words_results.extend(page_count)
    pdf_page_count_results.extend(page_results)
    member_validation_results.extend(member_results)
    salary_validation_results.extend(salary_results)
    transaction_validation_results.extend(acc_transaction_results)
    investment_validation_results.extend(investment_results)
    benefits_validation_results.extend(benefits_results)


valid_word_count_df = pd.DataFrame(page_words_results)
valid_pages_df = pd.DataFrame(pdf_page_count_results)
valid_member_df = pd.DataFrame(member_validation_results)
valid_salary_df = pd.DataFrame(salary_validation_results)
valid_contribution_df = pd.DataFrame(transaction_validation_results)
valid_investment_df = pd.DataFrame(investment_validation_results)
validate_benefits_df = pd.DataFrame(benefits_validation_results)


valid_word_count_df.to_csv("../post validations/word_counts_validation.csv", index=False)
valid_pages_df.to_csv("../post validations/page_counts_validation.csv", index=False)
valid_member_df.to_csv("../post validations/member_details_validation.csv", index=False)
valid_salary_df.to_csv("../post validations/salary_validation.csv", index=False)
valid_contribution_df.to_csv("../post validations/contributions_transactions_validation.csv", index=False)
valid_investment_df.to_csv("../post validations/investment_validation.csv", index=False)
validate_benefits_df.to_csv("../post validations/benefits_validation.csv", index=False)

8     0731447793
10    0731447793
18    0731447793
20    0731447793
28    0814373805
30    0814373805
38    0814373805
40    0814373805
48    0814373805
50    0814373805
58    0814373805
60    0814373805
Name: mobilephone, dtype: object
7            NaN
14           NaN
16           NaN
22           NaN
29    0815234014
31    0815234014
37    0815234014
44    0815234014
46    0815234014
52    0815234014
59    0815234014
61    0815234014
Name: mobilephone, dtype: object
9     0738276794
11    0738276794
17    0738276794
24    0738276794
26    0738065371
32    0738065371
39    0738065371
41    0738065371
47    0738065371
54    0738065371
56    0738065371
62    0738065371
Name: mobilephone, dtype: object
5            NaN
13           NaN
15           NaN
23           NaN
25    0719336182
33    0719336182
35    0719336182
43    0719336182
45    0719336182
53    0719336182
55    0719336182
63    0719336182
Name: mobilephone, dtype: object
6            NaN
12           NaN
19           NaN
2

In [46]:
pdf_data["full_text"]

"Mutualpark, Jan Smuts Drive, Pinelands, 7405. PO Box 728, Cape Town, 8000, South Africa.\nMember Queries: 0860 45 54 55, www.oldmutual.co.za\nSACCAWU NATIONAL PROVIDENT FUND\n 12 August 2026\nDear Member\nMEMBER BENEFIT STATEMENT\nOn behalf of the SACCAWU National Provident Fund (“the Fund”), we enclose your latest Member Benefit\nStatement showing the value of your Retirement Savings (accumulated credit) as at 31/12/2025, as well as\ndetails of the death, disability, spouse’s and family (funeral) benefits.\nWhen you read your statement, you should consider the following: Are my Retirement Savings enough to\nmeet my retirement needs? We recommend that you meet with a financial advisor to assist you in planning\nfor your retirement.\nAs a member of the Fund, you can be sure that your Retirement Savings will be managed according to the\nhighest standards. You can also take advantage of the tools and communication available from the Old\nMutual website:\n\x7f Visit our website at www.old